# Exemplo aplicado: rastreabilidade em uma clínica

Este notebook executa o mesmo cenário do texto: uma clínica ambulatorial que agenda consultas. Os IDs, nomes e relações são mantidos iguais ao `index.qmd`.

## 1. Requisitos, decisões, componentes e testes

O sistema recebe pedidos de consulta, verifica disponibilidade, confirma o horário e envia uma notificação.

In [ ]:
requirements = {
    'REQ-01': {'type': 'Funcional', 'name': 'Consultar horários disponíveis por especialidade', 'quality': 'Usabilidade'},
    'REQ-02': {'type': 'Funcional', 'name': 'Solicitar uma consulta e receber confirmação', 'quality': 'Modificabilidade'},
    'REQ-03': {'type': 'Funcional', 'name': 'Enviar confirmação por e-mail após o agendamento', 'quality': 'Interoperabilidade'},
    'REQ-04': {'type': 'Não funcional', 'name': 'Responder disponibilidade em até 2 segundos para 95% das consultas', 'quality': 'Desempenho'},
    'REQ-05': {'type': 'Não funcional', 'name': 'Permitir alterações somente a pacientes autenticados', 'quality': 'Segurança'},
}
decisions = {
    'DA-01': 'Separar o caso de uso de agenda em um serviço de aplicação com uma porta de disponibilidade',
    'DA-02': 'Persistir consultas em um repositório relacional com índice por profissional e horário',
    'DA-03': 'Publicar ConsultaAgendada e delegar a notificação a um adaptador de e-mail',
    'DA-04': 'Manter cache de leitura para horários disponíveis, invalidado após gravação',
    'DA-05': 'Proteger a API com token e autorização por paciente',
}
components = {
    'COMP-01': 'API de Agendamento', 'COMP-02': 'Serviço de Disponibilidade',
    'COMP-03': 'Repositório de Consultas', 'COMP-04': 'Publicador de Eventos',
    'COMP-05': 'Adaptador de E-mail', 'COMP-06': 'Autenticação e Cache',
}
tests = {
    'TST-01': 'Consultar horários disponíveis por especialidade',
    'TST-02': 'Solicitar horário livre, gravar consulta e retornar confirmação',
    'TST-03': 'Publicar evento e enviar e-mail após confirmação',
    'TST-04': 'Medir 95% das consultas de disponibilidade em até 2 segundos',
    'TST-05': 'Rejeitar requisição sem token e autorizar o próprio paciente',
}
print(f'{len(requirements)} requisitos, {len(decisions)} decisões, {len(components)} componentes e {len(tests)} testes')

## 2. Matriz de rastreabilidade

Cada linha registra links `realiza` entre um requisito e suas decisões, componentes e teste.

In [ ]:
traceability = [
    {'requirement': 'REQ-01', 'decisions': ['DA-01', 'DA-04'], 'components': ['COMP-01', 'COMP-02', 'COMP-06'], 'test': 'TST-01'},
    {'requirement': 'REQ-02', 'decisions': ['DA-01', 'DA-02'], 'components': ['COMP-01', 'COMP-02', 'COMP-03'], 'test': 'TST-02'},
    {'requirement': 'REQ-03', 'decisions': ['DA-03'], 'components': ['COMP-04', 'COMP-05'], 'test': 'TST-03'},
    {'requirement': 'REQ-04', 'decisions': ['DA-02', 'DA-04'], 'components': ['COMP-02', 'COMP-03', 'COMP-06'], 'test': 'TST-04'},
    {'requirement': 'REQ-05', 'decisions': ['DA-05'], 'components': ['COMP-01', 'COMP-06'], 'test': 'TST-05'},
]
for row in traceability: print(row)

## 3. Consultas de rastreabilidade

A consulta para frente parte de um requisito; a consulta para trás procura os requisitos que justificam um componente.

In [ ]:
def trace_forward(requirement_id):
    row = next(item for item in traceability if item['requirement'] == requirement_id)
    return row['decisions'] + row['components'] + [row['test']]
def trace_backward(component_id):
    return [item['requirement'] for item in traceability if component_id in item['components']]
print('REQ-02 para frente:', trace_forward('REQ-02'))
print('COMP-02 para trás:', trace_backward('COMP-02'))
assert trace_forward('REQ-02') == ['DA-01', 'DA-02', 'COMP-01', 'COMP-02', 'COMP-03', 'TST-02']
assert trace_backward('COMP-02') == ['REQ-01', 'REQ-02', 'REQ-04']

## 4. Alteração de requisito e impacto arquitetural

`REQ-04` muda de 2 segundos/95% para 500 ms/99%. O ID continua `REQ-04`, pois é uma evolução do mesmo requisito.

In [ ]:
changed_requirement = {'id': 'REQ-04', 'before': 'até 2 segundos para 95% das consultas', 'after': 'até 500 ms para 99% das consultas'}
impact = trace_forward(changed_requirement['id'])
expected_impact = ['DA-02', 'DA-04', 'COMP-02', 'COMP-03', 'COMP-06', 'TST-04']
print(f"{changed_requirement['id']}: {changed_requirement['before']} -> {changed_requirement['after']}")
print('Elementos afetados:', impact)
assert impact == expected_impact
assert 'DA-01' not in impact and 'COMP-01' not in impact and 'TST-02' not in impact
print('Análise de impacto validada.')

## 5. Conclusão

A matriz transforma uma mudança textual em um conjunto verificável de impactos. Para `REQ-04`, ela aponta `DA-02`, `DA-04`, `COMP-02`, `COMP-03`, `COMP-06` e `TST-04`.